# Binance RSI 策略回测

使用 backtrader 对 Binance 交易所的 BTC/USDT 和 ETH/USDT 进行 RSI 策略回测

In [2]:
import sys
sys.path.append("..")  # 添加父目录到路径中

# 现在可以导入了

In [1]:
import datetime as dt
import backtrader as bt
from backtrader_binance import BinanceStore
from ConfigBinance.Config import Config  # Configuration file
import warnings
import quantstats
import backtrader.analyzers as btanalyzers
from backtrader.filters import HeikinAshi
from backtrader.comminfo import CommInfoBase

ModuleNotFoundError: No module named 'ConfigBinance'

In [ ]:
class CommInfoFractional(CommInfoBase):
    params = (
        ('commission', 0.0004),  # 默认手续费
        ('mult', 1.0),          # 乘数
        ('margin', None),       # 保证金
    )
    
    def getsize(self, price, cash):
        return self.p.mult * cash / price

In [4]:
class RSIStrategy(bt.Strategy):
    params = (
        ('coin_target', ''),
        ('timeframe', ''),
        ('rsi_period', 14),        # RSI周期
        ('rsi_buy', 30),          # RSI买入阈值 
        ('rsi_sell', 70),         # RSI卖出阈值
        ('stop_loss_pct', 0.05),  # 止损比例
        ('btc_size', 0.0005),     # BTC默认交易量
        ('eth_size', 0.05),       # ETH默认交易量
    )

    def __init__(self):
        self.orders = {}
        for d in self.datas:
            self.orders[d._name] = None

        # 创建指标
        self.sma1 = {}
        self.sma2 = {} 
        self.rsi = {}
        for i in range(len(self.datas)):
            ticker = list(self.dnames.keys())[i]
            self.sma1[ticker] = bt.indicators.SMA(self.datas[i], period=8)
            self.sma2[ticker] = bt.indicators.SMA(self.datas[i], period=16)
            self.rsi[ticker] = bt.indicators.RSI(self.datas[i], period=self.p.rsi_period)
            
        # 记录每个交易对的买入价格,用于止损
        self.buy_price = {}

    def next(self):
        for data in self.datas:
            ticker = data._name
            status = data._state
            _interval = self.p.timeframe

            if status in [0, 1]:
                if status: _state = "False - History data"
                else: _state = "True - Live data"

                print('{} / {} [{}] - Open: {}, High: {}, Low: {}, Close: {}, Volume: {} - Live: {}'.format(
                    bt.num2date(data.datetime[0]),
                    data._name,
                    _interval,
                    data.open[0],
                    data.high[0],
                    data.low[0],
                    data.close[0],
                    data.volume[0],
                    _state,
                ))
                print(f'\t - {ticker} RSI : {self.rsi[ticker][0]}')

                coin_target = self.p.coin_target
                print(f"\t - Free balance: {self.broker.getcash()} {coin_target}")

                order = self.orders[data._name]
                if order and order.status == bt.Order.Submitted:
                    return

                position = self.getposition(data)
                
                # 检查是否需要止损
                if position:
                    unrealized_pnl = (data.close[0] - self.buy_price[ticker]) / self.buy_price[ticker]
                    if unrealized_pnl < -self.p.stop_loss_pct:
                        print(f"\t - 触发止损: {ticker} 当前亏损: {unrealized_pnl:.2%}")
                        self.orders[data._name] = self.close()
                        return

                if not position:  # 没有持仓
                    if order and order.status == bt.Order.Accepted:
                        print(f"\t - Cancel the order {order.p.tradeid} to buy {data._name}")
                        
                    # RSI低于买入阈值且SMA8在SMA16上方时买入
                    if (self.rsi[ticker][0] < self.p.rsi_buy and 
                        self.sma1[ticker][0] > self.sma2[ticker][0]):
                        
                        size = self.p.btc_size
                        if data._name == "ETHUSDT": 
                            size = self.p.eth_size

                        price = data.close[0]
                        self.buy_price[ticker] = price  # 记录买入价格

                        print(f" - buy {ticker} size = {size} at price = {price}")
                        self.orders[data._name] = self.buy(data=data, 
                                                         exectype=bt.Order.Limit,
                                                         price=price, 
                                                         size=size)
                        print(f"\t - The order has been submitted {self.orders[data._name].p.tradeid} to buy {data._name}")

                else:  # 有持仓
                    # RSI高于卖出阈值或SMA8跌破SMA16时卖出
                    if (self.rsi[ticker][0] > self.p.rsi_sell or
                        self.sma1[ticker][0] < self.sma2[ticker][0]):
                        print(f"\t - Sell signal triggered for {data._name}")
                        self.orders[data._name] = self.close()

    def notify_order(self, order):
        """Changing the status of the order"""
        order_data_name = order.data._name  # Name of ticker from order
        self.log(f'Order number {order.ref} {order.info["order_number"]} {order.getstatusname()} {"Buy" if order.isbuy() else "Sell"} {order_data_name} {order.size} @ {order.price}')
        if order.status == bt.Order.Completed:  # If the order is fully executed
            if order.isbuy():  # The order to buy
                self.log(f'Buy {order_data_name} @{order.executed.price:.2f}, Price {order.executed.value:.2f}, Commission {order.executed.comm:.2f}')
            else:  # The order to sell
                self.log(f'Sell {order_data_name} @{order.executed.price:.2f}, Price {order.executed.value:.2f}, Commission {order.executed.comm:.2f}')
            self.orders[order_data_name] = None  # Reset the order to enter the position

    def notify_trade(self, trade):
        """Changing the position status"""
        if trade.isclosed:  # If the position is closed
            self.log(f'Profit on a closed position {trade.getdataname()} Total={trade.pnl:.2f}, No commission={trade.pnlcomm:.2f}')

    def log(self, txt, dt=None):
        """Print string with date to the console"""
        dt = bt.num2date(self.datas[0].datetime[0]) if not dt else dt  # date or date of the current bar
        print(f'{dt.strftime("%d.%m.%Y %H:%M")}, {txt}')  # Print the date and time with the specified text to the console

In [ ]:
def run_backtest(
    strategy,
    start_date=None,
    end_date=None,
    symbols=['BTCUSDT', 'ETHUSDT'],
    timeframe='1m',
    compression=1,
    initial_cash=200000,
    commission=0.0015,
    plot=True,
    use_local_data=True,  # 新增参数：是否使用本地数据
    data_path='D:/CryptoData/data/spot',  # 新增参数：本地数据路径
    **strategy_params
):
    """
    整合的回测函数
    
    参数:
        strategy: 策略类
        start_date: 开始日期 (str或datetime对象，格式: 'YYYY-MM-DD' 或 'YYYY-MM-DD HH:MM:SS')
        end_date: 结束日期 (str或datetime对象，格式: 'YYYY-MM-DD' 或 'YYYY-MM-DD HH:MM:SS')
        symbols: 交易对列表
        timeframe: 时间周期 ('1m', '5m', '15m', '1h', '4h', '1d' 等)
        compression: K线周期压缩倍数
        initial_cash: 初始资金
        commission: 手续费率
        plot: 是否绘图
        strategy_params: 策略参数
    """
    # 处理日期参数
    if isinstance(start_date, str):
        try:
            start_date = dt.datetime.strptime(start_date, '%Y-%m-%d')
        except ValueError:
            start_date = dt.datetime.strptime(start_date, '%Y-%m-%d %H:%M:%S')
        start_date = start_date.replace(tzinfo=dt.UTC)
    
    if isinstance(end_date, str):
        try:
            end_date = dt.datetime.strptime(end_date, '%Y-%m-%d')
        except ValueError:
            end_date = dt.datetime.strptime(end_date, '%Y-%m-%d %H:%M:%S')
        end_date = end_date.replace(tzinfo=dt.UTC)
    
    # 如果没有设置开始日期，默认使用30天前
    if start_date is None:
        start_date = dt.datetime.now(dt.UTC) - dt.timedelta(days=30)
    
    # 如果没有设置结束日期，使用当前时间
    if end_date is None:
        end_date = dt.datetime.now(dt.UTC)
        
    # 时间周期映射
    timeframe_map = {
        '1m': (bt.TimeFrame.Minutes, 1),
        '5m': (bt.TimeFrame.Minutes, 5),
        '15m': (bt.TimeFrame.Minutes, 15),
        '30m': (bt.TimeFrame.Minutes, 30),
        '1h': (bt.TimeFrame.Minutes, 60),
        '4h': (bt.TimeFrame.Minutes, 240),
        '1d': (bt.TimeFrame.Days, 1),
    }
    
    tf, comp = timeframe_map.get(timeframe, (bt.TimeFrame.Minutes, 1))
    comp *= compression  # 应用压缩倍数
    
    cerebro = bt.Cerebro(quicknotify=True)
    
    # 设置资金和手续费
    cerebro.broker.setcash(initial_cash)
    comminfo = CommInfoFractional(commission=commission)
    cerebro.broker.addcommissioninfo(comminfo)
    
    # 创建 BinanceStore
    store = BinanceStore(
        api_key=Config.BINANCE_API_KEY,
        api_secret=Config.BINANCE_API_SECRET,
        coin_target='USDT',
        testnet=False
    )
    
    # 修改数据获取部分
    for symbol in symbols:
        if use_local_data:
            data = store.getlocaldata(
                timeframe=tf,
                compression=comp,
                dataname=symbol,
                start_date=start_date,
                end_date=end_date,
                datapath=data_path
            )
        else:
            data = store.getdata(
                timeframe=tf,
                compression=comp,
                dataname=symbol,
                start_date=start_date,
                end_date=end_date,
                LiveBars=False
            )
            
        if data is None:  # 如果获取数据失败
            print(f"无法获取 {symbol} 的数据，跳过")
            continue
            
        # 添加 HeikinAshi 过滤器
        if strategy_params.get('use_ha', False):
            data_ha = data.clone()
            data_ha.addfilter(HeikinAshi(data_ha))
            cerebro.adddata(data_ha, name=f"{symbol}_HA")
        cerebro.adddata(data, name=symbol)
    
    # 添加分析器
    cerebro.addanalyzer(bt.analyzers.PyFolio, _name='pyfolio')
    cerebro.addanalyzer(bt.analyzers.SharpeRatio, _name='sharpe')
    cerebro.addanalyzer(bt.analyzers.DrawDown, _name='drawdown')
    
    # 添加策略
    cerebro.addstrategy(strategy, **strategy_params)
    
    # 运行回测
    results = cerebro.run()
    strat = results[0]
    
    # 输出统计信息
    portfolio_value = cerebro.broker.getvalue()
    roi = (portfolio_value - initial_cash) / initial_cash * 100
    
    print(f'\n=== 回测结果 ===')
    print(f'初始资金: {initial_cash:,.2f}')
    print(f'最终资金: {portfolio_value:,.2f}')
    print(f'收益率: {roi:.2f}%')
    print(f'夏普比率: {strat.analyzers.sharpe.get_analysis()["sharperatio"]:.2f}')
    print(f'最大回撤: {strat.analyzers.drawdown.get_analysis()["max"]["drawdown"]:.2f}%')
    
    # 生成 quantstats 报告
    portfolio_stats = strat.analyzers.getbyname('pyfolio')
    returns, positions, transactions, gross_lev = portfolio_stats.get_pf_items()
    returns.index = returns.index.tz_convert(None)
    quantstats.reports.html(returns, output='backtest_report.html', title='Trading Strategy Analysis')
    
    # 绘图
    if plot:
        cerebro.plot(style='candlestick')
    
    return results

In [ ]:
if __name__ == '__main__':
    results = run_backtest(
        strategy=RSIStrategy,
        start_date='2024-01-01',
        end_date='2024-02-01',
        symbols=['BTCUSDT', 'ETHUSDT', 'SOLUSDT'],
        timeframe='15m',
        compression=1,
        initial_cash=200000,
        commission=0.0015,
        plot=True,
        use_local_data=True,  # 使用本地数据
        data_path='D:/CryptoData/data/spot',  # 指定本地数据路径
        # 策略参数
        rsi_period=14,
        rsi_buy=30,
        rsi_sell=70,
        stop_loss_pct=0.05,
        btc_size=0.0005,
        eth_size=0.05
    )